# Exploratory analysis

This notebook is for **exploration only** - all reusable logic lives in
`handwriting_intelligence` and is exercised by the test suite and CLI. We use it
here to sanity-check that datasets load correctly, that labels line up with the
documented conventions, and that the samples look upright.

Run it after acquiring data (see `docs/data.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from handwriting_intelligence.classes import build_class_set
from handwriting_intelligence.data.datasets import (
    load_mnist,
    locate_emnist_corpus,
    locate_mnist,
)

DATA_ROOT = Path('data')
print('MNIST found:', locate_mnist(DATA_ROOT) is not None)
print('EMNIST corpus found:', locate_emnist_corpus(DATA_ROOT))

In [ ]:
# --- MNIST: shapes, label range and a visual sample grid -------------------
mnist = load_mnist(DATA_ROOT, download=False)
images, labels = mnist['train_images'], mnist['train_labels']
print('images', images.shape, images.dtype)
print('labels', labels.shape, 'range', labels.min(), labels.max())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, idx in zip(axes.ravel(), range(16)):
    ax.imshow(images[idx], cmap='gray')
    ax.set_title(f'label={labels[idx]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()

In [ ]:
# --- Class balance --------------------------------------------------------
counts = np.bincount(labels, minlength=10)
plt.bar(range(10), counts)
plt.title('MNIST training class distribution')
plt.xlabel('digit'); plt.ylabel('count')
print('min/max class count:', counts.min(), counts.max())

In [ ]:
# --- Verify the EMNIST label conventions ----------------------------------
# Index 0..9 -> digits, 10..35 -> A-Z, 36..46 -> the non-merged lowercase letters.
cs = build_class_set('emnist_balanced')
print('classes:', cs.num_classes)
print('first 12 labels:', cs.labels[:12])
print('labels[36:47]:', cs.labels[36:47])

letters = build_class_set('emnist_letters')
print('letters has placeholder at 0:', letters.labels[0], '| classes:', letters.num_classes)

In [ ]:
# --- EMNIST corpus: upright orientation check ------------------------------
corpus_dir = locate_emnist_corpus(DATA_ROOT)
if corpus_dir is None:
    print('No local EMNIST corpus; skipping.')
else:
    from handwriting_intelligence.data.datasets import load_emnist_corpus

    arrays = load_emnist_corpus(corpus_dir, cache_dir=DATA_ROOT / 'cache', use_cache=True)
    images_e, labels_e = arrays['images'], arrays['labels']
    print('EMNIST images', images_e.shape, 'classes', int(labels_e.max()) + 1)
    fig, axes = plt.subplots(1, 8, figsize=(12, 2))
    for ax, cls in zip(axes, [0, 1, 10, 20, 36, 40, 45, 46]):
        idx = np.flatnonzero(labels_e == cls)[0]
        ax.imshow(images_e[idx], cmap='gray')
        ax.set_title(cs.labels[cls], fontsize=10)
        ax.axis('off')
    plt.tight_layout()